## NA2M - full pipeline: arm A vs arm B (GAMI-Net) vs arm C (NA2M)

In [1]:
import random
import numpy as np
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

SEED = 42
set_seed(SEED)

### Data

In [2]:
from na2m.data.data_utils import load_compas, preprocess

df = load_compas(r'C:\Users\maart\OneDrive\Documenten\Universiteit\Scriptie\python_repo\thesis-nam\datasets\raw\compas-scores-two-years.csv')
X, y, feature_meta = preprocess(df)

print(f'Samples: {X.shape[0]}, Features: {X.shape[1]}')
for fm in feature_meta:
    print(f'  {fm.name:25s} type={fm.type}')

Samples: 6907, Features: 6
  age                       type=num
  priors_count              type=num
  length_of_stay            type=num
  c_charge_degree           type=cat
  race                      type=cat
  sex                       type=cat


### Tune NA2M hyperparameters (arm B — used for both B and C)

In [3]:
import importlib.util
from pathlib import Path
from na2m.utils.config import load_na2m_search_config
from na2m.data.data_utils import split

_SEARCH_CONFIG = r'C:\Users\maart\OneDrive\Documenten\Universiteit\Scriptie\python_repo\thesis-nam\configs\compas_na2m_search.yaml'

# Load tune_fold without touching the na2m package namespace
_spec = importlib.util.spec_from_file_location(
    "tune_na2m_script",
    Path(_SEARCH_CONFIG).parent.parent / "scripts" / "na2m" / "tune_na2m.py",
)
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
tune_fold = _mod.tune_fold

fixed_params, search_space = load_na2m_search_config(_SEARCH_CONFIG)

# Single shared split for both the tuner and the arm training cells below
X_train, X_val, X_test, y_train, y_val, y_test = split(
    X, y, fixed_params['val_frac'], fixed_params['test_frac'], seed=SEED
)
print(f'Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}')

TUNED_PATH = Path(_SEARCH_CONFIG).parent / 'compas-scores-two-years_na2m_interactions_tuned.yaml'

tune_fold(
    fixed_params, search_space, feature_meta,
    X_train, y_train, X_val, y_val,
    TUNED_PATH,
    with_interactions=True,
)

[I 2026-06-13 21:52:05,229] A new study created in memory with name: fold_search


Train: 4834, Val: 691, Test: 1382
Epoch 1/100| Epoch loss = 0.7691 | best=0.0000
Epoch 100/100| Epoch loss = 0.6629 | best=0.7045
[stage2] FAST screen: top-10 pairs selected from 15 candidates
[stage2] Block-training 10 interaction subnets for 100 epochs...
Epoch 1/100| Epoch loss = 0.7795 | best=0.0000
Epoch 100/100| Epoch loss = 0.6376 | best=0.7242
[stage2] Block-train done. Best val metric: 0.7242
[stage2] Contribution ranking (variance, descending):
  (0,1)  var=0.015836
  (0,2)  var=0.010013
  (0,3)  var=0.003941
  (1,2)  var=0.003749
  (1,5)  var=0.003708
  (1,3)  var=0.002048
  (0,5)  var=0.001524
  (1,4)  var=0.000818
  (0,4)  var=0.000179
  (2,5)  var=0.000155
[stage2] Sweep baseline val loss (mains only): 0.64217
  pair (0, 1) — accepted  val_loss=0.62992
  pair (0, 2) — accepted  val_loss=0.62420
  pair (0, 3) — accepted  val_loss=0.62272
  pair (1, 2) — accepted  val_loss=0.61880
  pair (1, 5) — accepted  val_loss=0.61619
  pair (1, 3) — accepted  val_loss=0.61440
  pair (

[I 2026-06-13 21:54:13,859] Trial 0 finished with value: 0.7249410377358491 and parameters: {'activation': 'relu', 'lr': 0.004000034438560069, 'output_regularization': 0.005618501911221555, 'l2_regularization': 1.0177002838060295e-05, 'dropout': 0.9, 'feature_dropout': 0.2, 'clarity_regularization': 0.03326457008469609}. Best is trial 0 with value: 0.7249410377358491.


Early stopping at epoch 80
Epoch 1/100| Epoch loss = 0.6937 | best=0.0000
Early stopping at epoch 60
[stage2] FAST screen: top-10 pairs selected from 15 candidates
[stage2] Block-training 10 interaction subnets for 100 epochs...
Epoch 1/100| Epoch loss = 0.6510 | best=0.0000
Early stopping at epoch 100
[stage2] Block-train done. Best val metric: 0.7258
[stage2] Contribution ranking (variance, descending):
  (0,4)  var=0.057712
  (2,4)  var=0.042136
  (0,2)  var=0.010521
  (0,5)  var=0.005630
  (2,3)  var=0.004505
  (1,2)  var=0.001796
  (1,5)  var=0.001541
  (0,3)  var=0.000855
  (0,1)  var=0.000830
  (2,5)  var=0.000644
[stage2] Sweep baseline val loss (mains only): 0.61702
  pair (0, 4) — accepted  val_loss=0.62770
  pair (2, 4) — accepted  val_loss=0.61309
  pair (0, 2) — accepted  val_loss=0.61227
  pair (0, 5) — accepted  val_loss=0.61147
  pair (2, 3) — accepted  val_loss=0.61023
  pair (1, 2) — accepted  val_loss=0.61025
  pair (1, 5) — accepted  val_loss=0.60918
  pair (0, 3) —

[I 2026-06-13 21:56:30,561] Trial 1 finished with value: 0.7224561994609164 and parameters: {'activation': 'relu', 'lr': 0.012664861572560937, 'output_regularization': 0.006322703022403066, 'l2_regularization': 2.1455785149247782e-05, 'dropout': 0.05, 'feature_dropout': 0, 'clarity_regularization': 0.12263950927349009}. Best is trial 0 with value: 0.7249410377358491.


...

[I 2026-06-13 22:31:15,065] Trial 18 finished with value: 0.7237617924528303 and parameters: {'activation': 'relu', 'lr': 0.002474417775268537, 'output_regularization': 0.035196665795697385, 'l2_regularization': 5.734476687818357e-06, 'dropout': 0.1, 'feature_dropout': 0.1, 'clarity_regularization': 0.007193221155177187}. Best is trial 14 with value: 0.7275438005390835.


Epoch 1/100| Epoch loss = 0.7025 | best=0.0000
Epoch 100/100| Epoch loss = 0.6234 | best=0.7267
[stage2] FAST screen: top-10 pairs selected from 15 candidates
[stage2] Block-training 10 interaction subnets for 100 epochs...
Epoch 1/100| Epoch loss = 0.6269 | best=0.0000
Early stopping at epoch 100
[stage2] Block-train done. Best val metric: 0.7271
[stage2] Contribution ranking (variance, descending):
  (1,3)  var=0.005872
  (0,2)  var=0.002346
  (0,4)  var=0.001886
  (0,1)  var=0.000681
  (1,4)  var=0.000568
  (0,5)  var=0.000541
  (1,5)  var=0.000334
  (2,5)  var=0.000312
  (0,3)  var=0.000256
  (1,2)  var=0.000062
[stage2] Sweep baseline val loss (mains only): 0.61192
  pair (1, 3) — accepted  val_loss=0.60986
  pair (0, 2) — accepted  val_loss=0.60932
  pair (0, 4) — accepted  val_loss=0.60976
  pair (0, 1) — accepted  val_loss=0.60915
  pair (1, 4) — accepted  val_loss=0.60844
  pair (0, 5) — accepted  val_loss=0.60830
  pair (1, 5) — accepted  val_loss=0.60790
  pair (2, 5) — acce

[I 2026-06-13 22:32:31,400] Trial 19 finished with value: 0.7279144204851752 and parameters: {'activation': 'relu', 'lr': 0.0016585355511567399, 'output_regularization': 0.012736199949185295, 'l2_regularization': 3.413905766910163e-05, 'dropout': 0, 'feature_dropout': 0.2, 'clarity_regularization': 0.010234816291046687}. Best is trial 19 with value: 0.7279144204851752.


Best metric : 0.7279
Best config saved to C:\Users\maart\OneDrive\Documenten\Universiteit\Scriptie\python_repo\thesis-nam\configs\compas-scores-two-years_na2m_interactions_tuned.yaml


WindowsPath('C:/Users/maart/OneDrive/Documenten/Universiteit/Scriptie/python_repo/thesis-nam/configs/compas-scores-two-years_na2m_interactions_tuned.yaml')

### Config

In [4]:
from na2m.utils.config import load_na2m_config

config = load_na2m_config(str(TUNED_PATH))
config.top_m = 20
config.eta_prune = 0.01

print(config)

NA2MConfig(dataset_path='C:\\Users\\maart\\OneDrive\\Documenten\\Universiteit\\Scriptie\\python_repo\\thesis-nam\\datasets\\raw\\compas-scores-two-years.csv', num_units=64, hidden_sizes=[64, 32], activation='relu', dropout=0, inter_units=32, inter_hidden=[], feature_dropout=0.2, output_regularization=0.012736199949185295, l2_regularization=3.413905766910163e-05, clarity_regularization=0.010234816291046687, lr=0.0016585355511567399, decay_rate=0.995, val_frac=0.1, test_frac=0.2, seed=42, pool_val_frac=0.15, batch_size=1024, num_epochs=100, patience=50, val_check_interval=10, top_m=20, eta_prune=0.01, block_train_epochs=100, finetune_epochs=100, concurvity_filter=True, concurvity_threshold=0.5, k_folds=5, fold_seed=42, seeds=[0, 1, 2, 3, 4], grid_size=100, task='classification')


### Loaders

In [5]:
from nam.data.dataset import NAMDataset
from torch.utils.data import DataLoader

train_dataset = NAMDataset(X_train, y_train)
val_dataset   = NAMDataset(X_val, y_val)
pool_dataset  = NAMDataset(np.concatenate([X_train, X_val]), np.concatenate([y_train, y_val]))
test_dataset  = NAMDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=config.batch_size, shuffle=False)
pool_loader  = DataLoader(pool_dataset,  batch_size=config.batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=config.batch_size, shuffle=False)

print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}, Pool: {len(pool_dataset)}, Test: {len(test_dataset)}')

Train: 4834, Val: 691, Pool: 5525, Test: 1382


### Helpers

In [6]:
from na2m.models.na2m import NA2M
from na2m.training.fit_na2m import fit_na2m
from nam.training.metrics import auroc

def build_model():
    return NA2M(
        num_features=X.shape[1], feature_meta=feature_meta,
        num_units=config.num_units, hidden_sizes=config.hidden_sizes,
        dropout=config.dropout, feature_dropout=config.feature_dropout,
        activation=config.activation, inter_units=config.inter_units,
        inter_hidden=config.inter_hidden,
    )

def test_auroc(model):
    model.eval()
    logits_all, targets_all = [], []
    with torch.no_grad():
        for X_batch, y_batch, _ in test_loader:
            logits, _ = model(X_batch)
            logits_all.append(logits)
            targets_all.append(y_batch)
    return auroc(torch.cat(logits_all), torch.cat(targets_all))

def print_result(label, auc, result):
    pairs = result['active_pairs']
    print(f'{label}  AUROC={auc:.4f}  interactions={len(pairs)}')
    for j, k in pairs:
        print(f'  {feature_meta[j].name} x {feature_meta[k].name}')

### Arm A - mains only

In [7]:
set_seed(SEED)
model_a = build_model()

result_a = fit_na2m(
    model_a, train_loader, val_loader, pool_loader, config,
    with_interactions=False,
    with_concurvity_filter=False,
)

auc_a = test_auroc(model_a)
print_result('Arm A (mains only)', auc_a, result_a)

Epoch 1/100| Epoch loss = 0.7401 | best=0.0000
Epoch 100/100| Epoch loss = 0.6262 | best=0.7175
Arm A (mains only)  AUROC=0.7484  interactions=0


### Arm B - GAMI-Net (interactions, no concurvity filter)

In [8]:
set_seed(SEED)
model_b = build_model()

result_b = fit_na2m(
    model_b, train_loader, val_loader, pool_loader, config,
    with_interactions=True,
    with_concurvity_filter=False,
)

auc_b = test_auroc(model_b)
print_result('Arm B (GAMI-Net, no filter)', auc_b, result_b)

Epoch 1/100| Epoch loss = 0.7401 | best=0.0000
Epoch 100/100| Epoch loss = 0.6262 | best=0.7175
[stage2] FAST screen: top-15 pairs selected from 15 candidates
[stage2] Block-training 15 interaction subnets for 100 epochs...
Epoch 1/100| Epoch loss = 0.6905 | best=0.0000
Epoch 100/100| Epoch loss = 0.6158 | best=0.7232
[stage2] Block-train done. Best val metric: 0.7232
[stage2] Contribution ranking (variance, descending):
  (1,5)  var=0.004438
  (4,5)  var=0.002752
  (0,1)  var=0.002659
  (1,3)  var=0.002409
  (0,2)  var=0.002226
  (0,5)  var=0.002199
  (3,5)  var=0.001921
  (3,4)  var=0.001827
  (0,3)  var=0.001269
  (0,4)  var=0.000910
  (1,2)  var=0.000625
  (2,4)  var=0.000550
  (1,4)  var=0.000376
  (2,5)  var=0.000157
  (2,3)  var=0.000146
[stage2] Sweep baseline val loss (mains only): 0.61771
  pair (1, 5) — accepted  val_loss=0.61550
  pair (4, 5) — accepted  val_loss=0.61372
  pair (0, 1) — accepted  val_loss=0.61252
  pair (1, 3) — accepted  val_loss=0.61223
  pair (0, 2) — ac

### Arm C - NA2M (interactions + concurvity filter)

In [9]:
set_seed(SEED)
model_c = build_model()

result_c = fit_na2m(
    model_c, train_loader, val_loader, pool_loader, config,
    with_interactions=True,
    with_concurvity_filter=True,
)

auc_c = test_auroc(model_c)
print_result('Arm C (NA2M, with filter)', auc_c, result_c)

Epoch 1/100| Epoch loss = 0.7401 | best=0.0000
Epoch 100/100| Epoch loss = 0.6262 | best=0.7175
[stage2] FAST screen: top-15 pairs selected from 15 candidates
[stage2] Block-training 15 interaction subnets for 100 epochs...
Epoch 1/100| Epoch loss = 0.6905 | best=0.0000
Epoch 100/100| Epoch loss = 0.6158 | best=0.7232
[stage2] Block-train done. Best val metric: 0.7232
[stage2] Contribution ranking (variance, descending):
  (1,5)  var=0.004438
  (4,5)  var=0.002752
  (0,1)  var=0.002659
  (1,3)  var=0.002409
  (0,2)  var=0.002226
  (0,5)  var=0.002199
  (3,5)  var=0.001921
  (3,4)  var=0.001827
  (0,3)  var=0.001269
  (0,4)  var=0.000910
  (1,2)  var=0.000625
  (2,4)  var=0.000550
  (1,4)  var=0.000376
  (2,5)  var=0.000157
  (2,3)  var=0.000146
[stage2] Sweep baseline val loss (mains only): 0.61771
  pair (1, 5) — SKIPPED by concurvity gate
  pair (4, 5) — accepted  val_loss=0.61589
  pair (0, 1) — SKIPPED by concurvity gate
  pair (1, 3) — SKIPPED by concurvity gate
  pair (0, 2) — SK

### Comparison

In [10]:
print(f'Arm A (mains only):          AUROC={auc_a:.4f}  interactions=0')
print(f'Arm B (GAMI-Net, no filter): AUROC={auc_b:.4f}  interactions={len(result_b["active_pairs"])}')
print(f'Arm C (NA2M, with filter):   AUROC={auc_c:.4f}  interactions={len(result_c["active_pairs"])}')

Arm A (mains only):          AUROC=0.7484  interactions=0
Arm B (GAMI-Net, no filter): AUROC=0.7467  interactions=15
Arm C (NA2M, with filter):   AUROC=0.7475  interactions=5
